# AChE de novo design: ChemBERTa–ChemGPT generation, filtering, GNINA docking and GROMACS MD

Companion code for *"Generating Acetylcholinesterase Potential Inhibitors through Foundational Models Integration"* (Roeroe et al.).

This notebook merges four working notebooks into one readable pipeline:

| Part | What it does | Main outputs |
|---|---|---|
| 1 | Data acquisition and curation (ChEMBL, CHEMBL220, IC50 ≤ 1000 nM) | `data/active_reference_set.csv` |
| 2 | ChemBERTa → linear projection → ChemGPT generation (+ optional ablation baselines) | `generation/generated_selfies.csv` |
| 3 | SELFIES→SMILES, novelty, Lipinski (≤1 violation), KRFP Tanimoto filter, generative metrics | `generation/final_candidates.csv` |
| 4 | Receptor/ligand preparation, GNINA redocking and candidate docking (3 seeds) | `docking/*.csv` |
| 5 | GROMACS set-up, production, and trajectory analysis | `md/*/md_statistics_summary.csv` |

**Status of this merged version.** The code was assembled from notebooks that ran in Google Colab / JupyterHub. Heavy steps (ChEMBL download, generation of 34,990 sequences, docking of 79 ligands × 3 receptors × 3 seeds, 100-ns MD) were **not re-run** when merging. Run each part once and compare the printed numbers with the manuscript (expected values are printed next to the results). Cells marked `# EXPECTED:` give the number reported in the paper.

**Ligand numbering.** "Ligand 447 / 712 / 1130" in the paper is the `Original_Index` of the *reference molecule* (row in the 3,499-record active set) whose embedding conditioned the generation of that candidate. It is not a row number among the 79 candidates.

**Things done by hand outside this notebook:** ligand parameterisation with the ACPYPE web server (https://www.bio2byte.be/acpype/, AM1-BCC charges), SwissADME and ProTox 3.0 predictions, Discovery Studio 2D/3D interaction diagrams.

## 0. Environment

Versions recorded in the original notebook logs (per stage):

| Stage | Software |
|---|---|
| Generation | Python 3.12.12, PyTorch 2.12.0+cu130, Transformers 4.36.0, SELFIES 1.0.3, RDKit 2026.03.2, ChEMBL web resource client 0.10.9, NumPy 1.26.4, pandas 2.2.2 |
| Filtering / metrics | RDKit 2026.03.4, SELFIES 1.0.3, PyTorch 2.11.0+cpu, NumPy 2.0.2, pandas 2.2.2, matplotlib 3.10.0, seaborn 0.13.2, tqdm 4.67.3, fcd_torch |
| Docking | GNINA 1.3.2 (CUDA 12.8 build), Open Babel 3.1.1, Discovery Studio 25.1.0 |
| MD | GROMACS 2021.4 (Ubuntu package: pdb2gmx, solvate, genion, minimisation, NVT, NPT), GROMACS 2026.1 (20-ns production and all analysis), GROMACS 2025.4 conda-forge (100-ns production of Ligand 712), PDBFixer 1.12.0, ACPYPE web server |

Use `requirements.txt` for the Python packages. GNINA, Open Babel and GROMACS are system tools.

In [ ]:
# Optional: install pinned Python packages (uncomment on a fresh machine).
# !pip install torch==2.12.0 transformers==4.36.0 "numpy==1.26.4" pandas==2.2.2 \
#     rdkit==2026.3.4 selfies==1.0.3 chembl_webresource_client==0.10.9 \
#     fcd_torch tqdm matplotlib seaborn pdbfixer openmm

In [ ]:
import os, re, json, glob, random, shutil, subprocess, sys, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import selfies as sf
from tqdm import tqdm
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem, Descriptors, Crippen, rdMolDescriptors, rdMolAlign
from rdkit.Chem.Scaffolds import MurckoScaffold

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("RDKit   :", rdBase.rdkitVersion)
print("SELFIES :", sf.__version__ if hasattr(sf, "__version__") else "see pip show selfies")

In [ ]:
# ------------------------------- CONFIGURATION -------------------------------
WORKDIR = Path("ache_workdir")
DATA, GEN, DOCK, MD = (WORKDIR / d for d in ("data", "generation", "docking", "md"))
for p in (DATA, GEN, DOCK, MD):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42                               # global seed used for generation (set before model loading)

# Data
TARGET_ID = "CHEMBL220"                 # human acetylcholinesterase (UniProt P22303)
IC50_CUTOFF_NM = 1000                   # pIC50 >= 6

# Models
CHEMBERTA_ID = "DeepChem/ChemBERTa-10M-MLM"   # 3 layers, hidden 384, 12 heads
CHEMGPT_ID   = "ncfrey/ChemGPT-19M"           # GPT-Neo, 24 layers, hidden 256
N_PER_REF = 10                          # sequences per reference molecule (3,499 x 10 = 34,990)
DECODING = dict(max_new_tokens=150, do_sample=True, top_k=50, top_p=0.90, temperature=1.0)

# Filtering
TC_MIN = 0.60                           # KRFP Tanimoto lower bound
TC_EXCLUDE_AT_OR_ABOVE = 0.90           # the original run used < 0.90; only one molecule (Tc = 1.00) is affected
MAX_LIPINSKI_VIOLATIONS = 1             # classical limits: MW <= 500, LogP <= 5, HBD <= 5, HBA <= 10

# Docking
RECEPTORS = {"7E3H": "E20", "4EY6": "GNT", "7XN1": "THA"}   # PDB id -> co-crystallised ligand residue name
DOCK_SEEDS = [0, 1, 2]
EXHAUSTIVENESS = 16
AUTOBOX_ADD = 10                        # Angstrom margin around the reference ligand
# NOTE: in the original redocking cells GNINA's default margin was used and only seed 0 was run.
# Re-run redocking with the settings below and check that Table 8 is reproduced.
REDOCK_AUTOBOX_ADD = AUTOBOX_ADD

## Part 1. Data acquisition and curation (ChEMBL)

In [ ]:
from chembl_webresource_client.new_client import new_client

# Confirm that the target is the human single-protein entry.
hits = pd.DataFrame.from_dict(new_client.target.search("acetylcholinesterase"))
print(hits.loc[hits.target_chembl_id == TARGET_ID, ["target_chembl_id", "organism", "pref_name", "target_type"]])

activity = new_client.activity
records = activity.filter(target_chembl_id=TARGET_ID).filter(standard_type="IC50")
raw = pd.DataFrame.from_dict(records)
raw.to_csv(DATA / "chembl_ic50_raw.csv", index=False)
print("Raw IC50 records:", len(raw))              # EXPECTED: 9731 (ChEMBL 36, retrieved 19 Oct 2025)

In [ ]:
# Curation funnel (Table 1): remove records without SMILES or value, then exact duplicates
# (identical ChEMBL ID + SMILES + standard value). Records with the same molecule but a different IC50 are kept.
n_raw = len(raw)
clean = raw.dropna(subset=["standard_value", "canonical_smiles"])
n_missing = n_raw - len(clean)

keep_cols = ["molecule_chembl_id", "canonical_smiles", "standard_value", "standard_relation", "standard_units"]
df = (clean[keep_cols]
      .drop_duplicates(subset=["molecule_chembl_id", "canonical_smiles", "standard_value"])
      .rename(columns={"molecule_chembl_id": "ChEMBL_ID", "canonical_smiles": "SMILES", "standard_value": "IC50_nM"})
      .reset_index(drop=True))
df["IC50_nM"] = pd.to_numeric(df["IC50_nM"], errors="coerce")

print("Initial records          :", n_raw)                    # EXPECTED: 9731
print("Missing SMILES/value     :", n_missing)                # EXPECTED: 1288
print("Exact duplicates removed :", len(clean) - len(df))     # EXPECTED: 602
print("Curated records          :", len(df))                  # EXPECTED: 7841

active = df[df["IC50_nM"] <= IC50_CUTOFF_NM].reset_index(drop=True)
print("Active reference records :", len(active))             # EXPECTED: 3499
print("Distinct molecules       :", active["ChEMBL_ID"].nunique())   # EXPECTED: 2938

# Audit of the records used as the active set (paper: all '=' and 'nM').
print(active["standard_relation"].value_counts(dropna=False).to_dict(),
      active["standard_units"].value_counts(dropna=False).to_dict())

active.to_csv(DATA / "active_reference_set.csv", index=False)
sample_df = active          # name used in the original notebooks

## Part 2. ChemBERTa → linear projection → ChemGPT generation

Both pretrained models are frozen. The 384-d mean-pooled ChemBERTa embedding (input: SMILES) is mapped to 256-d by an
**untrained** `torch.nn.Linear` layer (PyTorch default initialisation) and **prepended as one extra position before the start token**
through `inputs_embeds`. The projection layer is kept in float16 on GPU, as in the original run.

Reproducibility note: one global seed is set once before the models are loaded. Resuming an interrupted run changes the random stream,
so a resumed run will not be bit-identical to an uninterrupted one.

In [ ]:
from transformers import AutoTokenizer, RobertaModel, AutoModelForCausalLM

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proj_dtype = torch.float16 if device.type == "cuda" else torch.float32

# Order matters for exact reproducibility: seed -> ChemBERTa -> ChemGPT -> projection layer.
bert_tok = AutoTokenizer.from_pretrained(CHEMBERTA_ID)
bert = RobertaModel.from_pretrained(CHEMBERTA_ID).to(device).eval()

gpt_tok = AutoTokenizer.from_pretrained(CHEMGPT_ID)
gpt_tok.pad_token = gpt_tok.eos_token
gpt = AutoModelForCausalLM.from_pretrained(CHEMGPT_ID).to(device).eval()

projection = torch.nn.Linear(bert.config.hidden_size, gpt.config.hidden_size).to(device).to(proj_dtype).eval()
print("Projection:", bert.config.hidden_size, "->", gpt.config.hidden_size, "| dtype", proj_dtype)

START_ID = next(i for i in (gpt_tok.bos_token_id, gpt_tok.eos_token_id, gpt_tok.pad_token_id, 0) if i is not None)

@torch.no_grad()
def chemberta_embedding(smiles):
    enc = bert_tok(smiles, return_tensors="pt", padding=True, truncation=True).to(device)
    return bert(**enc).last_hidden_state.mean(dim=1).to(proj_dtype)      # (1, 384), mean pooling

@torch.no_grad()
def generate_conditioned(smiles, n=N_PER_REF):
    cond = projection(chemberta_embedding(smiles)).unsqueeze(1)           # (1, 1, 256)
    start = gpt.transformer.wte(torch.tensor([[START_ID]], device=device))
    embeds = torch.cat([cond.to(start.dtype), start], dim=1)              # [conditioning, <start>]
    mask = torch.ones(embeds.shape[:2], device=device)
    out = gpt.generate(inputs_embeds=embeds, attention_mask=mask, num_return_sequences=n,
                       pad_token_id=gpt_tok.pad_token_id, **DECODING)
    return [s.strip() for s in gpt_tok.batch_decode(out, skip_special_tokens=True)]

In [ ]:
# Generation loop with simple checkpointing (every 10 reference molecules).
GEN_CSV = GEN / "generated_selfies.csv"
done = pd.read_csv(GEN_CSV) if GEN_CSV.exists() else pd.DataFrame()
start_idx = int(done["Original_Index"].max()) + 1 if len(done) else 0

rows = []
for idx in tqdm(range(start_idx, len(sample_df)), desc="Generating"):
    ref = sample_df.iloc[idx]
    for k, s in enumerate(generate_conditioned(ref["SMILES"]), start=1):
        rows.append({"ChEMBL_ID_source": ref["ChEMBL_ID"], "SMILES_source": ref["SMILES"],
                     "Generated_SELFIES": s, "Generation_ID": k, "Original_Index": idx})
    if (idx + 1) % 10 == 0 or idx + 1 == len(sample_df):
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_csv(GEN_CSV, index=False)
        rows = []

gen_df = pd.read_csv(GEN_CSV)
print("Generated sequences:", len(gen_df))        # EXPECTED: 34990

### 2b. Ablation baselines (Section 2.4) — *reconstructed, verify against your original ablation notebook*

The original ablation notebooks were not among the files merged here, so the two baselines below are re-written from the description
in Section 2.4 (same decoding parameters; the reference SELFIES is given to ChemGPT as the prompt). Confirm that they match the code
that produced Table 4 before publishing, or replace them with the original cells.

In [ ]:
@torch.no_grad()
def generate_baseline(smiles, mode, n=N_PER_REF):
    # mode = "chemgpt_only": SELFIES tokens passed as input_ids
    # mode = "pytorch_embedding": same tokens mapped by the frozen pretrained wte table and passed as inputs_embeds
    selfies_str = sf.encoder(smiles)
    ids = gpt_tok(selfies_str, return_tensors="pt", truncation=True, max_length=256).input_ids.to(device)
    if mode == "chemgpt_only":
        out = gpt.generate(input_ids=ids, attention_mask=torch.ones_like(ids), num_return_sequences=n,
                           pad_token_id=gpt_tok.pad_token_id, **DECODING)
    elif mode == "pytorch_embedding":
        emb = torch.nn.Embedding.from_pretrained(gpt.transformer.wte.weight.detach().clone(), freeze=True)
        e = emb(ids)
        out = gpt.generate(inputs_embeds=e, attention_mask=torch.ones(e.shape[:2], device=device),
                           num_return_sequences=n, pad_token_id=gpt_tok.pad_token_id, **DECODING)
    else:
        raise ValueError(mode)
    return [s.strip() for s in gpt_tok.batch_decode(out, skip_special_tokens=True)]

# Example (one reference molecule):
# generate_baseline(sample_df.loc[0, "SMILES"], "chemgpt_only")

## Part 3. Conversion, novelty, Lipinski and KRFP filtering

Counting convention (Table 5):
*converted* = SELFIES→SMILES succeeded (33,789; 96.57 %); *valid* = also RDKit-sanitisable (26,598; 76.02 %, the validity reported in Table 4).

In [ ]:
# 3.1 SELFIES -> SMILES
df = gen_df.copy()
df["selfies_cleaned"] = df["Generated_SELFIES"].astype(str).str.replace(" ", "").str.strip()

def decode_selfies(s):
    try:
        sf.len_selfies(s)
        return sf.decoder(s)
    except Exception:
        return None

df["Converted_SMILES"] = [decode_selfies(s) for s in tqdm(df["selfies_cleaned"], desc="SELFIES->SMILES")]
df_conv = df[df["Converted_SMILES"].notna()].copy()
print("Converted:", len(df_conv), f"({100*len(df_conv)/len(df):.2f} %)")     # EXPECTED: 33789 (96.57 %)

In [ ]:
# 3.2 Canonical (stereo-agnostic) SMILES, RDKit validity, novelty vs ChEMBL, internal duplicates
def canon(smiles, isomeric=False):
    if not isinstance(smiles, str):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=isomeric)
    except Exception:
        return None

df_conv["canonical_generated"] = [canon(s) for s in tqdm(df_conv["Converted_SMILES"], desc="canonicalise")]
df_valid = df_conv.dropna(subset=["canonical_generated"]).copy()
print("RDKit-valid:", len(df_valid), f"({100*len(df_valid)/len(df):.2f} %)")   # EXPECTED: 26598 (76.02 %)

# ChEMBL structures: download chembl_36_chemreps.txt.gz from the ChEMBL FTP site and place it in data/.
CHEMBL_REPS = DATA / "chembl_chemreps.txt.gz"
# The original run compared against ChEMBL's canonical_smiles as provided (isomeric). Set True to strip stereo
# from ChEMBL as well (slower, ~2.4 M molecules) so both sides are stereo-agnostic.
STRIP_CHEMBL_STEREO = False
chembl = pd.read_csv(CHEMBL_REPS, sep="\t", usecols=["canonical_smiles"])["canonical_smiles"].dropna().astype(str).str.strip()
if STRIP_CHEMBL_STEREO:
    chembl = chembl.map(lambda s: canon(s, isomeric=False)).dropna()
chembl_set = set(chembl)

df_novel = df_valid[~df_valid["canonical_generated"].isin(chembl_set)]
df_novel_unique = df_novel.drop_duplicates(subset=["canonical_generated"], keep="first")
print("ChEMBL matches    :", len(df_valid) - len(df_novel))          # EXPECTED: 11
print("Internal duplicates:", len(df_novel) - len(df_novel_unique))  # EXPECTED: 118
print("Novel and unique  :", len(df_novel_unique))                   # EXPECTED: 26469

In [ ]:
# 3.3 Lipinski Rule of Five, at most one violation (classical limits, i.e. a violation is MW > 500, LogP > 5, HBD > 5, HBA > 10)
def lipinski_row(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series({"MolWt": None, "LogP": None, "HDonors": None, "HAcceptors": None, "TPSA": None,
                          "Lipinski_Violations": None, "Pass_Lipinski": False})
    mw, lp = Descriptors.MolWt(mol), Crippen.MolLogP(mol)
    hbd, hba = Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol)
    v = int(mw > 500) + int(lp > 5) + int(hbd > 5) + int(hba > 10)
    return pd.Series({"MolWt": mw, "LogP": lp, "HDonors": hbd, "HAcceptors": hba,
                      "TPSA": rdMolDescriptors.CalcTPSA(mol), "Lipinski_Violations": v,
                      "Pass_Lipinski": v <= MAX_LIPINSKI_VIOLATIONS})

props = df_novel_unique["canonical_generated"].apply(lipinski_row)
df_lip = pd.concat([df_novel_unique, props], axis=1)
df_lip = df_lip[df_lip["Pass_Lipinski"]].copy()
print("Pass Lipinski:", len(df_lip))                                 # EXPECTED: 7110

**Klekota–Roth fingerprints (KRFP).** The notebooks used a local `KRFingerprints.py` + `krfp.json` (4,860-bit Klekota–Roth substructure keys).
Place both files in `ache_workdir/krfp/` (use the implementation you originally used and check its licence before redistributing it).

In [ ]:
sys.path.insert(0, str(WORKDIR / "krfp"))
from KRFingerprints import KRFingerprints          # local module (not on PyPI)

def to_bitvect(bits):
    bv = DataStructs.ExplicitBitVect(4860)
    for i, v in enumerate(bits):
        if v:
            bv.SetBit(i)
    return bv

def kr_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return to_bitvect(KRFingerprints.GenerateKRFingerprints(mol, count=False, output_type="list", verbose=False)[0])
    except Exception:
        return None

# Reference fingerprints: the 3,499 active records (duplicated molecules do not change the maximum Tanimoto).
ref_fps = [fp for fp in (kr_fp(s) for s in tqdm(sample_df["SMILES"], desc="reference KRFP")) if fp is not None]

max_sim = []
for smi in tqdm(df_lip["canonical_generated"], desc="Tanimoto"):
    fp = kr_fp(smi)
    max_sim.append(max(DataStructs.BulkTanimotoSimilarity(fp, ref_fps)) if fp is not None else np.nan)
df_lip["Tanimoto_Max_KR"] = max_sim

# Sensitivity of the threshold (Table 6)
sens = pd.DataFrame({"Tc_threshold": [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.90]})
sens["retained"] = [int((df_lip["Tanimoto_Max_KR"] >= t).sum()) for t in sens["Tc_threshold"]]
sens["percent_of_lipinski_set"] = (100 * sens["retained"] / len(df_lip)).round(2)
print(sens)      # EXPECTED retained: 4299, 905, 278, 80, 22, 5, 1

final = df_lip[(df_lip["Tanimoto_Max_KR"] >= TC_MIN) & (df_lip["Tanimoto_Max_KR"] < TC_EXCLUDE_AT_OR_ABOVE)].copy()
final["SMILES_Final"] = final["canonical_generated"]
final = final.sort_values("Tanimoto_Max_KR", ascending=False).reset_index(drop=True)

# Candidate names use the Original_Index of the conditioning reference molecule ("Ligand 712" = Original_Index 712).
final["Ligand_ID"] = [f"Ligand_{i}" for i in final["Original_Index"]]
dup = final["Ligand_ID"].duplicated(keep=False)
final.loc[dup, "Ligand_ID"] += "_g" + final.loc[dup, "Generation_ID"].astype(str)
final.to_csv(GEN / "final_candidates.csv", index=False)
print("Final candidates:", len(final))               # EXPECTED: 79

### 3b. Generative metrics (Table 4)

Validity, uniqueness, novelty, internal diversity (1 − mean Tanimoto of 500,000 sampled pairs, Klekota–Roth fingerprints, 14,000-molecule
samples, five seeds), scaffold diversity (unique/ring-containing Murcko scaffolds) and FCD (10 bootstrap subsamples of 10,000 generated molecules
against the active reference set).

Novelty is computed against the **canonical, stereo-agnostic** reference structures. The original notebook compared against the raw ChEMBL SMILES
strings of the reference set; re-run this cell and confirm the value in Table 4 (99.54 % for the integrated model).

In [ ]:
validity = 100 * len(df_valid) / len(df)
uniqueness = 100 * df["Generated_SELFIES"].nunique() / len(df)
ref_canon = set(filter(None, (canon(s) for s in sample_df["SMILES"])))
novelty = 100 * len(set(df_valid["canonical_generated"]) - ref_canon) / len(df_valid)
print(f"Validity {validity:.2f} % | Uniqueness {uniqueness:.2f} % | Novelty {novelty:.2f} %")   # EXPECTED 76.02 / 99.99 / 99.54

# Internal diversity on sampled pairs
valid_fps = [fp for fp in (kr_fp(s) for s in tqdm(df_valid["canonical_generated"], desc="KRFP (all valid)")) if fp is not None]

def internal_diversity(fps, n_pairs=500_000, seed=42):
    rng = np.random.default_rng(seed)
    i = rng.integers(0, len(fps), n_pairs); j = rng.integers(0, len(fps), n_pairs)
    keep = i != j
    sims = [DataStructs.TanimotoSimilarity(fps[a], fps[b]) for a, b in zip(i[keep], j[keep])]
    return 1 - float(np.mean(sims))

divs = []
for sd in [42, 7, 123, 2024, 99]:
    idx = random.Random(sd).sample(range(len(valid_fps)), min(14000, len(valid_fps)))
    divs.append(internal_diversity([valid_fps[k] for k in idx], seed=sd))
print(f"Internal diversity {100*np.mean(divs):.2f} % (SD {100*np.std(divs, ddof=1):.2f})")      # EXPECTED ~73.05 %

# Scaffold diversity
scaf = []
for s in df_valid["canonical_generated"]:
    m = Chem.MolFromSmiles(s)
    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m is not None else None
    if sc:
        scaf.append(sc)
print(f"Scaffold diversity {100*len(set(scaf))/len(scaf):.2f} %")                                # EXPECTED ~82.72 %

In [ ]:
# FCD (fcd_torch): reference statistics fixed, generated side bootstrapped
from fcd_torch import FCD
fcd = FCD(device="cuda:0" if torch.cuda.is_available() else "cpu", n_jobs=1)
ref_smiles = [s for s in sample_df["SMILES"].dropna() if Chem.MolFromSmiles(s) is not None]
gen_smiles = df_valid["canonical_generated"].tolist()
scores = []
for i in range(10):
    sub = random.Random(SEED + i).sample(gen_smiles, min(10000, len(gen_smiles)))
    scores.append(fcd(ref_smiles, sub))
print(f"FCD {np.mean(scores):.2f} (SD {np.std(scores):.2f})")                                    # EXPECTED ~23.60

### 3c. Token-level attention (Figure 5)

**Check before reusing Figure 5.** The original cell for Figure 5 loaded `DeepChem/ChemBERTa-77M-MLM` (6 layers, hidden 768) and fed it the generated
**SELFIES strings**, mapping BPE-token attention back to SELFIES tokens by character offsets. That is not the encoder used in the pipeline
(`ChemBERTa-10M-MLM`, which receives SMILES). The cell below computes the same quantity (final layer, head-averaged `[CLS]` attention) with the pipeline
encoder on the SMILES of the reference molecules that conditioned the final candidates.

In [ ]:
bert_attn = RobertaModel.from_pretrained(CHEMBERTA_ID, output_attentions=True).to(device).eval()

@torch.no_grad()
def cls_attention(smiles):
    enc = bert_tok(smiles, return_tensors="pt", truncation=True).to(device)
    last = bert_attn(**enc).attentions[-1]                 # (1, heads, seq, seq)
    cls_row = last.mean(dim=1)[0, 0].cpu().numpy()          # head-averaged attention paid by [CLS]
    return bert_tok.convert_ids_to_tokens(enc["input_ids"][0]), cls_row

ref_for_candidates = sample_df.loc[final["Original_Index"], "SMILES"].tolist()
rows = []
for k, smi in enumerate(ref_for_candidates, start=1):
    toks, att = cls_attention(smi)
    d = {}
    for t, a in zip(toks, att):
        d.setdefault(t, []).append(a)
    rows.append({"Candidate": k, **{t: float(np.mean(v)) for t, v in d.items()}})
att_df = pd.DataFrame(rows).set_index("Candidate").fillna(0.0)
att_top = att_df[att_df.sum().sort_values(ascending=False).index[:30]]
print("Overall mean attention:", float(att_df.values[att_df.values > 0].mean()))

import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(14, 14)); sns.heatmap(att_top, cmap="Blues", linewidths=0.4)
plt.xlabel("ChemBERTa SMILES tokens"); plt.ylabel("Candidate"); plt.tight_layout()
plt.savefig(GEN / "attention_heatmap.png", dpi=300); plt.show()

## Part 4. Docking with GNINA

*Receptors*: chain A ATOM records only, hydrogens added with Open Babel (`-h`, no pH-based protonation); waters, ions and the co-crystallised ligand removed.
*Reference ligands*: HETATM records of the co-crystallised ligand (chain A), converted to SDF with Open Babel.
*Candidates*: 3D conformers by RDKit ETKDGv3 (random coordinates as fallback), MMFF94 optimisation, translated to the reference-ligand centroid.
*Box*: `--autobox_ligand <reference ligand> --autobox_add 10`; `--exhaustiveness 16`; `--cnn_scoring refinement`; candidates with `--num_modes 1`, three seeds (0, 1, 2).

In [ ]:
def sh(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, text=True, **kw)

def prepare_receptor(pdb_id, lig_resname):
    d = DOCK / "receptor_prep"; d.mkdir(exist_ok=True)
    pdb = d / f"{pdb_id}.pdb"
    if not pdb.exists():
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", pdb)
    lines = pdb.read_text().splitlines()
    rec = [l for l in lines if l.startswith("ATOM") and len(l) > 21 and l[21] == "A"]
    lig = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == lig_resname and l[21] == "A"]
    (d / f"{pdb_id}_rec_raw.pdb").write_text("\n".join(rec) + "\n")
    (d / f"{pdb_id}_lig_raw.pdb").write_text("\n".join(lig) + "\n")
    sh(["obabel", d / f"{pdb_id}_rec_raw.pdb", "-O", d / f"{pdb_id}_rec_clean.pdb", "-h"])
    sh(["obabel", d / f"{pdb_id}_lig_raw.pdb", "-O", d / f"{pdb_id}_lig_clean.sdf", "-h"])

for pid, res in RECEPTORS.items():
    prepare_receptor(pid, res)

In [ ]:
# Candidate ligands: 3D conformers and translation to each reference-ligand centroid
def make_3d(smiles, name, seed=SEED):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    p = AllChem.ETKDGv3(); p.useRandomCoords = True; p.randomSeed = seed   # seed added for reproducibility (not set originally)
    if AllChem.EmbedMolecule(mol, p) == -1:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        pass
    mol.SetProp("_Name", name)
    return mol

prep = DOCK / "ligand_prep"; prep.mkdir(exist_ok=True)
mols = [make_3d(s, n) for s, n in zip(final["SMILES_Final"], final["Ligand_ID"])]
print("3D conversion failures:", sum(m is None for m in mols))
mols = [m for m in mols if m is not None]

def centroid(mol):
    return mol.GetConformer().GetPositions().mean(axis=0)

for pid in RECEPTORS:
    ref = next(Chem.SDMolSupplier(str(DOCK / "receptor_prep" / f"{pid}_lig_clean.sdf"), removeHs=False))
    c_ref = centroid(ref)
    print(pid, "box centre (A):", np.round(c_ref, 3))
    # EXPECTED: 7E3H (-43.568, 37.916, -30.274); 4EY6 (-10.234, -43.518, 30.446); 7XN1 (47.998, -40.266, -30.211)
    w = Chem.SDWriter(str(prep / f"ligands_{pid}_centered.sdf"))
    for m in mols:
        m2 = Chem.Mol(m)
        conf = m2.GetConformer()
        shift = c_ref - centroid(m2)
        for i, pos in enumerate(conf.GetPositions() + shift):
            conf.SetAtomPosition(i, pos.tolist())
        w.write(m2)
    w.close()

# Approximate search-box edges: reference-ligand extent + 2 x margin (check against the GNINA documentation for your version).
for pid in RECEPTORS:
    ref = next(Chem.SDMolSupplier(str(DOCK / "receptor_prep" / f"{pid}_lig_clean.sdf"), removeHs=False))
    xyz = ref.GetConformer().GetPositions()
    print(pid, "approx. box edges (A):", np.round(xyz.max(0) - xyz.min(0) + 2 * AUTOBOX_ADD, 1))

In [ ]:
def run_gnina(receptor_id, ligand_sdf, seed, out_sdf, autobox_add, num_modes=None):
    cp = DOCK / "receptor_prep"
    cmd = ["gnina", "-r", cp / f"{receptor_id}_rec_clean.pdb", "-l", ligand_sdf,
           "--autobox_ligand", cp / f"{receptor_id}_lig_clean.sdf", "--autobox_add", autobox_add,
           "--seed", seed, "--exhaustiveness", EXHAUSTIVENESS, "--cnn_scoring", "refinement",
           "--device", 0, "-o", out_sdf, "--log", str(out_sdf) + ".log"]
    if num_modes:
        cmd += ["--num_modes", num_modes]
    sh(cmd)

out = DOCK / "poses"; out.mkdir(exist_ok=True)

# 4.1 Redocking of the co-crystallised inhibitors (Table 8)
for pid in RECEPTORS:
    ref = DOCK / "receptor_prep" / f"{pid}_lig_clean.sdf"
    for sd in DOCK_SEEDS:
        run_gnina(pid, ref, sd, out / f"redock_{pid}_seed{sd}.sdf", REDOCK_AUTOBOX_ADD)

# 4.2 Candidate docking: 79 ligands x 3 receptors x 3 seeds (GPU recommended)
for pid in RECEPTORS:
    for sd in DOCK_SEEDS:
        run_gnina(pid, prep / f"ligands_{pid}_centered.sdf", sd, out / f"cand_{pid}_seed{sd}.sdf", AUTOBOX_ADD, num_modes=1)

In [ ]:
# 4.3 Redocking RMSD: symmetry-aware, heavy atoms, NO superposition (docked pose compared in the receptor frame)
def redock_table():
    rows = []
    for pid in RECEPTORS:
        ref = Chem.RemoveHs(next(Chem.SDMolSupplier(str(DOCK / "receptor_prep" / f"{pid}_lig_clean.sdf"), removeHs=False)))
        for sd in DOCK_SEEDS:
            for mode, m in enumerate(Chem.SDMolSupplier(str(out / f"redock_{pid}_seed{sd}.sdf"), removeHs=False), start=1):
                if m is None:
                    continue
                rows.append({"receptor": pid, "seed": sd, "mode": mode,
                             "affinity": float(m.GetProp("minimizedAffinity")),
                             "CNNscore": float(m.GetProp("CNNscore")),
                             "RMSD": rdMolAlign.CalcRMS(Chem.RemoveHs(m), ref)})
    return pd.DataFrame(rows)

redock = redock_table()
top = redock[redock["mode"] == 1]                  # GNINA mode 1 = top CNN-ranked pose
print(top.groupby("receptor")[["affinity", "CNNscore", "RMSD"]].agg(["mean", "std"]).round(2))
print("Success (RMSD <= 2 A):", (top["RMSD"] <= 2.0).groupby(top["receptor"]).mean().to_dict())
# EXPECTED (Table 8, seed 0): 7E3H 0.54, 4EY6 0.39, 7XN1 2.62 (fails)
redock.to_csv(DOCK / "redocking_all_modes.csv", index=False)

In [ ]:
# 4.4 Candidate results: mean +- SD over three seeds and mean pairwise (best-aligned, heavy-atom) RMSD between the three poses
import itertools
recs = []
for pid in RECEPTORS:
    poses = {}
    for sd in DOCK_SEEDS:
        for m in Chem.SDMolSupplier(str(out / f"cand_{pid}_seed{sd}.sdf"), removeHs=False):
            if m is None:
                continue
            poses.setdefault(m.GetProp("_Name"), {})[sd] = m
    for name, by_seed in poses.items():
        aff = [float(m.GetProp("minimizedAffinity")) for m in by_seed.values()]
        cnn = [float(m.GetProp("CNNscore")) for m in by_seed.values()]
        ok = [a < 0 for a in aff]                           # positive affinities are treated as failed seeds
        hm = [Chem.RemoveHs(m) for m in by_seed.values()]
        rm = [rdMolAlign.GetBestRMS(Chem.Mol(a), Chem.Mol(b)) for a, b in itertools.combinations(hm, 2)]
        recs.append({"Ligand_ID": name, "receptor": pid, "n_seeds": len(aff), "n_failed_positive": int(len(ok) - sum(ok)),
                     "affinity_mean": np.mean([a for a, o in zip(aff, ok) if o]) if any(ok) else np.nan,
                     "affinity_sd": np.std([a for a, o in zip(aff, ok) if o], ddof=1) if sum(ok) > 1 else np.nan,
                     "cnn_mean": np.mean([c for c, o in zip(cnn, ok) if o]) if any(ok) else np.nan,
                     "pose_rmsd_mean": np.mean(rm) if rm else np.nan})
dock_summary = pd.DataFrame(recs)
dock_summary.to_csv(DOCK / "candidate_docking_summary.csv", index=False)
dock_summary[dock_summary["Ligand_ID"].isin(["Ligand_447", "Ligand_712", "Ligand_1130"])].round(3)
# EXPECTED (Table 9), e.g. Ligand_712 / 7E3H: affinity -8.227 +- 0.458, CNN 0.860, pose RMSD 0.736

### 4.5 Choosing candidates for MD

Candidates were compared in **7E3H and 4EY6** (the two receptors whose redocking passed; 7XN1 is not used for selection) on three criteria evaluated over the three seeds:
lowest mean minimised affinity, high CNN score consistent with that affinity, and a low mean pairwise pose RMSD (stable poses). The cell below turns this into explicit ranks.
Replace the aggregation rule with the exact one you applied, and report the ranks (Supplementary Table).

In [ ]:
sel = dock_summary[dock_summary["receptor"].isin(["7E3H", "4EY6"])].copy()
sel["r_aff"] = sel.groupby("receptor")["affinity_mean"].rank(ascending=True)
sel["r_cnn"] = sel.groupby("receptor")["cnn_mean"].rank(ascending=False)
sel["r_rmsd"] = sel.groupby("receptor")["pose_rmsd_mean"].rank(ascending=True)
sel["mean_rank"] = sel[["r_aff", "r_cnn", "r_rmsd"]].mean(axis=1)
ranking = sel.groupby("Ligand_ID")["mean_rank"].mean().sort_values()
ranking.to_csv(DOCK / "md_selection_ranking.csv")
print(ranking.head(10)); print(ranking.reindex(["Ligand_447", "Ligand_712", "Ligand_1130"]))

## Part 5. Molecular dynamics with GROMACS

Protocol (Section 2.8): AMBER99SB-ILDN protein, ligand topology from the **ACPYPE web server** (AM1-BCC charges, GAFF), TIP3P water, rhombic dodecahedron box with 1.0 nm solute–box distance,
neutralisation plus 0.15 M NaCl, steepest-descent minimisation, 100 ps NVT and 100 ps NPT (310 K, 1 bar; position restraints on the solute), then unrestrained production (2 fs, LINCS on H-bonds, PME,
1.2 nm cut-offs, V-rescale thermostat τ = 0.1 ps, Parrinello–Rahman barostat τ = 2.0 ps, compressibility 4.5×10⁻⁵ bar⁻¹).
Frames are written every 10 ps and energies every 2 ps. NVT velocities were generated with `gen_seed = -1` (random), so runs are not bit-reproducible.

Manual step: upload the ligand (docked pose in the 7E3H frame) to the ACPYPE web server and place `<name>_GMX.gro` and `<name>_GMX.itp` in `md/<ligand>/`.

All GROMACS calls below go through one helper, so the executable is explicit. (In the original Colab notebook `%%bash` cells did not expand `{GMX}`, so minimisation and equilibration silently ran with the
system `gmx` 2021.4 instead of the 2026.1 build.) Set `GMX` to the build you use and record `gmx --version`.

In [ ]:
GMX = os.environ.get("GMX", "gmx")
LIG = "Ligand_712"                         # ligand under simulation (Original_Index 712)
MD_DIR = MD / LIG; MD_DIR.mkdir(exist_ok=True)
LIG_GRO, LIG_ITP = MD_DIR / "ligand_GMX.gro", MD_DIR / "ligand_GMX.itp"   # from the ACPYPE web server
PROD_NS = 20                                # 20 for the comparative runs, 100 for the extended run of Ligand 712
MDRUN_GPU = ["-nb", "gpu", "-pme", "gpu", "-bonded", "gpu", "-update", "gpu"]   # drop on CPU-only machines

def gmx(*args, stdin=None, cwd=MD_DIR):
    cmd = [GMX, *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, input=stdin, text=True, cwd=cwd, check=True)

gmx("--version")

In [ ]:
# 5.1 Protein preparation: 7E3H chain A -> PDBFixer (missing atoms; hydrogens at pH 7.4) -> pdb2gmx (AMBER99SB-ILDN, TIP3P, -ignh)
from pdbfixer import PDBFixer
from openmm.app import PDBFile

shutil.copy(DOCK / "receptor_prep" / "7E3H_rec_raw.pdb", MD_DIR / "protein_raw.pdb")
fixer = PDBFixer(filename=str(MD_DIR / "protein_raw.pdb"))
fixer.findMissingResidues(); fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.findMissingAtoms(); fixer.addMissingAtoms(); fixer.addMissingHydrogens(7.4)
with open(MD_DIR / "protein_fixed.pdb", "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# pdb2gmx rebuilds hydrogens (-ignh) and applies its default protonation states for the chosen force field.
gmx("pdb2gmx", "-f", "protein_fixed.pdb", "-o", "protein.gro", "-p", "topol.top", "-i", "posre.itp",
    "-ignh", "-ff", "amber99sb-ildn", "-water", "tip3p")

In [ ]:
# 5.2 Merge protein and ligand coordinates; add the ligand to the topology
def read_gro(path):
    lines = Path(path).read_text().splitlines()
    n = int(lines[1]); return lines[2:2 + n], lines[2 + n]

p_atoms, box = read_gro(MD_DIR / "protein.gro")
l_atoms, _ = read_gro(LIG_GRO)
last_atom = int(p_atoms[-1][15:20]); res_id = int(p_atoms[-1][0:5]) + 1
merged = list(p_atoms)
for k, line in enumerate(l_atoms, start=1):
    x, y, z = float(line[20:28]), float(line[28:36]), float(line[36:44])
    merged.append(f"{res_id:5d}{line[5:10].strip():<5}{line[10:15].strip():>5}{last_atom + k:5d}{x:8.3f}{y:8.3f}{z:8.3f}")
(MD_DIR / "complex.gro").write_text("Protein-ligand complex\n" + f"{len(merged):5d}\n" + "\n".join(merged) + "\n" + box + "\n")
print("complex atoms:", len(merged))

# Topology: include the ligand .itp after the force-field include and append the molecule to [ molecules ].
itp_text = LIG_ITP.read_text()
lig_molname = re.search(r"\[\s*moleculetype\s*\]\s*\n(?:;.*\n)*\s*(\S+)", itp_text).group(1)
top = (MD_DIR / "topol.top").read_text().splitlines()
pos = next(i for i, l in enumerate(top) if l.startswith('#include') and 'forcefield.itp' in l)
top.insert(pos + 1, f'#include "{LIG_ITP.name}"')
top.append(f"{lig_molname:<20}1")
(MD_DIR / "topol.top").write_text("\n".join(top) + "\n")
shutil.copy(LIG_ITP, MD_DIR / LIG_ITP.name)

In [ ]:
# 5.3 Box, solvation, neutralisation (+0.15 M NaCl)
MDP_IONS = "integrator=steep\nemtol=1000.0\nemstep=0.01\nnsteps=50000\ncutoff-scheme=Verlet\ncoulombtype=PME\nrcoulomb=1.0\nrvdw=1.0\npbc=xyz\n"
(MD_DIR / "ions.mdp").write_text(MDP_IONS)

gmx("editconf", "-f", "complex.gro", "-o", "box.gro", "-bt", "dodecahedron", "-d", "1.0")
gmx("solvate", "-cp", "box.gro", "-cs", "spc216.gro", "-o", "solvated.gro", "-p", "topol.top")
gmx("grompp", "-f", "ions.mdp", "-c", "solvated.gro", "-p", "topol.top", "-o", "ions.tpr", "-maxwarn", "15")
gmx("genion", "-s", "ions.tpr", "-o", "neutralized.gro", "-p", "topol.top",
    "-pname", "NA", "-nname", "CL", "-neutral", "-conc", "0.15", stdin="SOL\n")

# System composition for the Supplementary Table (atoms, waters, ions)
print((MD_DIR / "topol.top").read_text().split("[ molecules ]")[-1])
print("atoms:", int((MD_DIR / "neutralized.gro").read_text().splitlines()[1]))

In [ ]:
# 5.4 Parameter files
COMMON = '''
cutoff-scheme = Verlet
nstlist       = 10
coulombtype   = PME
rcoulomb      = 1.2
rvdw          = 1.2
DispCorr      = EnerPres
pbc           = xyz
'''
(MD_DIR / "em.mdp").write_text("integrator=steep\nemtol=1000.0\nemstep=0.01\nnsteps=50000\n" + COMMON)

THERMO = "tcoupl=V-rescale\ntc-grps=Protein_UNK Water_and_ions\ntau_t=0.1 0.1\nref_t=310 310\nconstraints=h-bonds\nconstraint_algorithm=lincs\n"
BARO = "pcoupl=Parrinello-Rahman\npcoupltype=isotropic\ntau_p=2.0\nref_p=1.0\ncompressibility=4.5e-5\n"
(MD_DIR / "nvt.mdp").write_text("define=-DPOSRES\nintegrator=md\nnsteps=50000\ndt=0.002\n" + THERMO +
                                "pcoupl=no\ngen_vel=yes\ngen_temp=310\ngen_seed=-1\n" + COMMON)
(MD_DIR / "npt.mdp").write_text("define=-DPOSRES\nintegrator=md\nnsteps=50000\ndt=0.002\n" + THERMO + BARO +
                                "continuation=yes\ngen_vel=no\n" + COMMON)
nsteps = int(PROD_NS * 1000 / 0.002)
(MD_DIR / "md.mdp").write_text(f"integrator=md\nnsteps={nsteps}\ndt=0.002\nnstxout-compressed=5000\nnstenergy=1000\nnstlog=1000\n" +
                               THERMO + BARO + "continuation=yes\ngen_vel=no\n" + COMMON.replace("nstlist       = 10", "nstlist       = 20"))
print("production steps:", nsteps)

In [ ]:
# 5.5 Index, minimisation, NVT, NPT, production
# Group numbers depend on the system: check the listing printed by make_ndx (original: 1 = Protein, 13 = UNK).
gmx("make_ndx", "-f", "neutralized.gro", "-o", "index.ndx", stdin="1 | 13\nq\n")

gmx("grompp", "-f", "em.mdp", "-c", "neutralized.gro", "-p", "topol.top", "-o", "em.tpr", "-maxwarn", "15")
gmx("mdrun", "-deffnm", "em")
gmx("grompp", "-f", "nvt.mdp", "-c", "em.gro", "-r", "em.gro", "-p", "topol.top", "-n", "index.ndx", "-o", "nvt.tpr", "-maxwarn", "15")
gmx("mdrun", "-deffnm", "nvt", *MDRUN_GPU)
gmx("grompp", "-f", "npt.mdp", "-c", "nvt.gro", "-r", "nvt.gro", "-t", "nvt.cpt", "-p", "topol.top", "-n", "index.ndx", "-o", "npt.tpr", "-maxwarn", "15")
gmx("mdrun", "-deffnm", "npt", *MDRUN_GPU)
gmx("grompp", "-f", "md.mdp", "-c", "npt.gro", "-t", "npt.cpt", "-p", "topol.top", "-n", "index.ndx", "-o", "md.tpr", "-maxwarn", "15")
# -noappend writes md.partNNNN.* so that an interrupted run can be resumed with the same command plus -cpi md.cpt.
gmx("mdrun", "-s", "md.tpr", "-deffnm", "md", "-noappend", *MDRUN_GPU)

### 5.6 Trajectory analysis (Tables 11–12, Figures 7–9)

* Segments `md.partNNNN.xtc` from restarts are concatenated in time order (empty parts are skipped). Check that the merged time axis is continuous (`gmx check`).
* **Protein backbone RMSD**: least-squares fit on the backbone, RMSD of the backbone, reference = `md.tpr` (start of production).
* **Ligand RMSD** as reported in the paper: fit on the ligand, RMSD of the ligand, i.e. the ligand's *internal conformational* change. It does not capture translation or rotation in the pocket.
  The additional output `rmsd_ligand_proteinfit.xvg` (fit on the backbone, RMSD of the ligand) and the protein–ligand COM distance describe positional drift and should be inspected before interpreting a ligand "shift".
* Protein–ligand hydrogen-bond counts, minimum distance and contact numbers are computed but were not reported in the manuscript; residue-level occupancies were not computed.

In [ ]:
parts = sorted(p for p in MD_DIR.glob("md.part*.xtc") if p.stat().st_size > 0)
print(len(parts), "trajectory parts")
gmx("trjcat", "-f", *[p.name for p in parts], "-o", "md_full.xtc")
gmx("check", "-f", "md_full.xtc")

gmx("trjconv", "-s", "md.tpr", "-f", "md_full.xtc", "-o", "md_center.xtc", "-pbc", "mol", "-center", stdin="Protein\nSystem\n")

# Group names (original runs used the numbers 4 = Backbone and 13 = UNK from index.ndx)
gmx("rms", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "rmsd_backbone.xvg", "-tu", "ns", stdin="Backbone\nBackbone\n")
gmx("rms", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "rmsd_ligand.xvg", "-tu", "ns", stdin="UNK\nUNK\n")
gmx("rms", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "rmsd_ligand_proteinfit.xvg", "-tu", "ns", stdin="Backbone\nUNK\n")
gmx("rmsf", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "rmsf.xvg", "-res", stdin="Protein\n")
gmx("rmsf", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "rmsf_ligand.xvg", stdin="UNK\n")
gmx("gyrate", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "gyrate.xvg", stdin="Protein\n")
gmx("sasa", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "sasa.xvg", "-or", "residue_sasa.xvg", stdin="Protein\n")
gmx("sasa", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "ligand_sasa.xvg", stdin="UNK\n")
gmx("distance", "-s", "md.tpr", "-f", "md_center.xtc", "-select", 'com of group "Protein" plus com of group "UNK"', "-oall", "dist_com.xvg")
gmx("mindist", "-s", "md.tpr", "-f", "md_center.xtc", "-od", "mindist.xvg", "-on", "contacts.xvg", stdin="Protein\nUNK\n")
gmx("hbond", "-s", "md.tpr", "-f", "md_center.xtc", "-num", "hbnum.xvg", stdin="Protein\nUNK\n")

edr = sorted(p.name for p in MD_DIR.glob("md.part*.edr") if p.stat().st_size > 0)
gmx("eneconv", "-f", *edr, "-o", "md_full.edr")
for term, fname in [("Temperature", "temperature"), ("Pressure", "pressure"), ("Potential", "potential"),
                    ("Total-Energy", "total_energy"), ("Density", "density")]:
    gmx("energy", "-f", "md_full.edr", "-o", f"{fname}.xvg", stdin=f"{term}\n0\n")

In [ ]:
def read_xvg(path):
    rows = [l.split() for l in Path(path).read_text().splitlines() if l and l[0] not in "#@"]
    return pd.DataFrame([[float(r[0]), float(r[1])] for r in rows if len(r) >= 2], columns=["x", "value"])

items = [("Protein backbone RMSD (nm)", "rmsd_backbone.xvg"), ("Ligand RMSD (nm)", "rmsd_ligand.xvg"),
         ("Ligand RMSD, protein-fit (nm)", "rmsd_ligand_proteinfit.xvg"),
         ("Protein residue RMSF (nm)", "rmsf.xvg"), ("Ligand RMSF (nm)", "rmsf_ligand.xvg"),
         ("Protein SASA (nm2)", "sasa.xvg"), ("Ligand SASA (nm2)", "ligand_sasa.xvg"), ("Rg (nm)", "gyrate.xvg"),
         ("Protein-ligand COM distance (nm)", "dist_com.xvg"), ("Protein-ligand H-bonds", "hbnum.xvg"),
         ("Temperature (K)", "temperature.xvg"), ("Pressure (bar)", "pressure.xvg"),
         ("Potential energy (kJ/mol)", "potential.xvg"), ("Total energy (kJ/mol)", "total_energy.xvg"),
         ("Density (kg/m3)", "density.xvg")]
summary = pd.DataFrame([{"Parameter": n, "Mean": read_xvg(MD_DIR / f).value.mean(), "SD": read_xvg(MD_DIR / f).value.std()}
                        for n, f in items if (MD_DIR / f).exists()])
summary.to_csv(MD_DIR / "md_statistics_summary.csv", index=False)
summary.round(4)
# EXPECTED, Ligand 712 100 ns (Table 12): protein RMSD 0.1869 +- 0.0161; ligand RMSD 0.2474 +- 0.0344;
# protein RMSF 0.1164 +- 0.0751; ligand RMSF 0.1225 +- 0.0461; Rg 2.3057 +- 0.0079; SASA 213.8144 +- 3.4443

In [ ]:
# Time series figure (RMSD, Rg, SASA, potential energy)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for a, (title, f) in zip(ax.ravel(), [("RMSD (nm)", None), ("Rg (nm)", "gyrate.xvg"), ("SASA (nm2)", "sasa.xvg"), ("Potential energy (kJ/mol)", "potential.xvg")]):
    if f is None:
        for lab, fn in [("Protein backbone", "rmsd_backbone.xvg"), ("Ligand", "rmsd_ligand.xvg")]:
            d = read_xvg(MD_DIR / fn); a.plot(d.x, d.value, label=lab, lw=0.8)
        a.legend()
    else:
        d = read_xvg(MD_DIR / f); a.plot(d.x, d.value, lw=0.6)
    a.set_title(title); a.set_xlabel("Time (ns)" if f != "potential.xvg" else "Time (ps)")
plt.tight_layout(); plt.savefig(MD_DIR / "md_overview.png", dpi=300); plt.show()